# Part 2 · Word2Vec 词向量

Part 1 的 BoW 有两个硬伤：向量极度稀疏，且 `awful` 和 `terrible` 是两个
互不相关的维度。Word2Vec 换了个思路：

> **一个词的含义，由它周围经常出现哪些词决定。**（分布式假设）

于是把「预测上下文」当作训练任务，用语料自己给自己造标签：

```
CBOW      : [the, movie, was, ___, boring]  →  预测 "really"
Skip-gram : "really"  →  预测周围的 the / movie / was / boring
```

没有任何人工标注，标签直接来自文本本身——这就是**自监督学习**。
今天 BERT 的掩码预测、GPT 的下一词预测，用的是同一套逻辑，只是把
这里的浅层网络换成了 Transformer。


## Setup

gensim 的 Word2Vec 是 Cython 多线程 CPU 实现，**不使用 GPU**。开加速器只会白占配额。


In [ ]:
import csv
import re
from html import unescape
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

HTML_TAG = re.compile(r"<[^>]+>")
NON_LETTER = re.compile(r"[^a-zA-Z]")
SENTENCE_END = re.compile(r"(?<=[.!?])\s+")
STOP_WORDS = frozenset(ENGLISH_STOP_WORDS)
RANDOM_STATE = 42


def find_data_dir() -> Path:
    for candidate in [Path("/kaggle/input/word2vec-nlp-tutorial"), Path("data"), Path("../data")]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("请用 Add Input 挂载 word2vec-nlp-tutorial 竞赛数据")


def read_tsv(data_dir: Path, stem: str) -> pd.DataFrame:
    # quoting=QUOTE_NONE：影评正文含大量引号，按默认规则解析会把字段粘连。
    for suffix in (".tsv", ".tsv.zip"):
        path = data_dir / f"{stem}{suffix}"
        if path.exists():
            return pd.read_csv(path, header=0, delimiter="\t", quoting=csv.QUOTE_NONE)
    raise FileNotFoundError(f"{data_dir} 下找不到 {stem}")


def review_to_words(raw_review, remove_stopwords: bool = True) -> list[str]:
    text = NON_LETTER.sub(" ", HTML_TAG.sub(" ", unescape(str(raw_review))))
    words = text.lower().split()
    if remove_stopwords:
        return [w for w in words if w not in STOP_WORDS]
    return words


def clean_review(raw_review, remove_stopwords: bool = True) -> str:
    return " ".join(review_to_words(raw_review, remove_stopwords))


DATA_DIR = find_data_dir()
print("数据目录:", DATA_DIR)


In [ ]:
import gensim
from gensim.models import Word2Vec
print("gensim:", gensim.__version__)


## 1 · 把无标注数据也用上

关键点：Word2Vec 训练**不看 sentiment 标签**。所以竞赛里那 50,000 条
无标注影评是免费的额外语料，一起用能让词向量学得更好。

25,000 标注 + 50,000 无标注 = 75,000 条影评，约 1,780 万词。


In [ ]:
labeled = read_tsv(DATA_DIR, "labeledTrainData")
unlabeled = read_tsv(DATA_DIR, "unlabeledTrainData")
print(f"标注 {len(labeled):,} 条 + 无标注 {len(unlabeled):,} 条")


## 2 · 分句，而不是分词

Word2Vec 的输入是「句子列表」，每句是词列表。为什么要分句？因为上下文
窗口不该跨越句子边界——上一句的句尾和下一句的句首在语义上没有关系。

两点和 Part 1 不同：

1. **保留停用词**。它们也是上下文窗口的一部分，删掉会让词与词的距离失真。
2. 用标点正则分句，而非 `nltk.punkt`（避免运行时下载模型）。


In [ ]:
def review_to_sentences(raw_review) -> list[list[str]]:
    text = HTML_TAG.sub(" ", unescape(str(raw_review)))
    out = []
    for chunk in SENTENCE_END.split(text):
        # 注意 remove_stopwords=False
        words = review_to_words(chunk, remove_stopwords=False)
        if words:
            out.append(words)
    return out


print(review_to_sentences(labeled.loc[0, "review"])[:2])


In [ ]:
sentences = []
for review in labeled["review"]:
    sentences.extend(review_to_sentences(review))
for review in unlabeled["review"]:
    sentences.extend(review_to_sentences(review))

print(f"{len(sentences):,} 句，{sum(len(s) for s in sentences):,} 词")


## 3 · 训练

沿用原教程的超参数：

| 参数 | 值 | 作用 |
|---|---|---|
| `vector_size` | 300 | 每个词的向量维度 |
| `min_count` | 40 | 词频下限，滤掉拼写错误和长尾词 |
| `window` | 10 | 上下文窗口半径 |
| `sample` | 1e-3 | 高频词降采样，避免 the/and 主导训练 |
| `sg` | 0 | 0 = CBOW（快），1 = Skip-gram（低频词更准） |

> API 变更：gensim 4.x 把 `size` 改名 `vector_size`、`iter` 改名 `epochs`，
> 词向量从 `model[word]` 改为 `model.wv[word]`。原教程的写法在 4.x 会报错。


In [ ]:
import logging
logging.basicConfig(format="%(asctime)s %(levelname)s %(message)s", level=logging.INFO)

model = Word2Vec(
    sentences,
    vector_size=300, min_count=40, window=10, sample=1e-3,
    sg=0, epochs=5, workers=4, seed=RANDOM_STATE,
)
print(f"\n词表 {len(model.wv):,} 个词 × {model.wv.vector_size} 维")


## 4 · 检查它到底学到了什么

这是最有说服力的一步：看看向量空间里的邻居是不是真的语义相近。


In [ ]:
for word in ["awful", "brilliant", "actress", "france"]:
    if word in model.wv:
        neighbours = ", ".join(w for w, _ in model.wv.most_similar(word, topn=6))
        print(f"{word:10s} → {neighbours}")


In [ ]:
# 挑出不属于同一类的词
for group in (["kitchen", "bedroom", "bathroom", "france"],
              ["awful", "terrible", "dreadful", "great"]):
    print(f"{group} → {model.wv.doesnt_match(group)}")


In [ ]:
# 向量算术：语义关系被编码成了空间中的方向
print("king - man + woman =")
for word, score in model.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3):
    print(f"   {word}  ({score:.3f})")


## 5 · 把 300 维压到 2 维看一眼

用 PCA 降维后画出来。注意这只是一个粗糙的投影——300 维里的大部分结构
在 2 维平面上必然丢失，所以图上「看起来近」不等于真的近。


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

groups = {
    "情感-正": ["excellent", "wonderful", "brilliant", "superb", "fantastic"],
    "情感-负": ["awful", "terrible", "horrible", "dreadful", "boring"],
    "角色":   ["actor", "actress", "director", "writer", "producer"],
    "国家":   ["france", "italy", "germany", "japan", "russia"],
}
words = [w for group in groups.values() for w in group if w in model.wv]
coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(model.wv[words])

fig, ax = plt.subplots(figsize=(8, 6))
index = 0
for (label, group), colour in zip(groups.items(), ["#2a9d8f", "#e76f51", "#4c72b0", "#9c6ade"]):
    present = [w for w in group if w in model.wv]
    block = coords[index:index + len(present)]
    ax.scatter(block[:, 0], block[:, 1], s=70, color=colour, label=label)
    for (x, y), word in zip(block, present):
        ax.annotate(word, (x, y), fontsize=9, xytext=(4, 4), textcoords="offset points")
    index += len(present)
ax.legend(prop={"size": 9})
ax.set(title="Word2Vec 向量空间的 PCA 投影")
plt.tight_layout()
plt.show()


## 6 · 保存模型给 Part 3 用

词向量是**可复用资产**：训练一次，喂给任意下游任务。这一点是 BoW 做不到的——
某个数据集上统计出的 5,000 词词表没法迁移到别处。预训练范式就是从这里开始的。


In [ ]:
output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("models")
output_dir.mkdir(parents=True, exist_ok=True)
model.save(str(output_dir / "word2vec_300d.model"))
print("已保存到", output_dir / "word2vec_300d.model")


## 小结

| | Bag of Words | Word2Vec |
|---|---|---|
| 维度 | 5,000（稀疏） | 300（稠密） |
| 每一维的含义 | 一个具体的词 | 无法单独解读 |
| 近义词 | 完全正交 | 空间中相邻 |
| 是否需要标注 | 需要标签才能训分类器 | 自监督，不需标签 |
| 可迁移 | 否 | 是 |

**下一步（Part 3）**：词向量是「词」的表示，分类器要的是「整条影评」的表示。
怎么从词向量拼出文档向量？
